In [2]:
!pip install sentencepiece
!pip install transformers
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 44.3 MB/s eta 0:00:00


#Seed 설정

In [3]:
# Seed 고정
import random
import numpy as np
import torch
import os

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

#데이터 로드 (Colab Drive)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/LLM수업/자연어 특강/5차년도_2차.csv",
    encoding="cp949"
)

Mounted at /content/drive


# Label Encoding

In [5]:
label_map = {
    "fear": 0,
    "surprise": 1,
    "angry": 2,
    "sadness": 3,
    "neutral": 4,
    "happiness": 5,
    "disgust": 6
}

df["y"] = df["상황"].map(label_map)

x_col = '발화문'
y_col = 'y'

input_data = df[[x_col, y_col]]
input_data.head()

,발화문,y
0,헐! 나 이벤트에 당첨 됐어.,5
1,내가 좋아하는 인플루언서가 이벤트를 하더라고. 그래서 그냥 신청 한번 해봤지.,5
2,"한 명 뽑는 거였는데, 그게 바로 내가 된 거야.",5
3,"당연히 마음에 드는 선물이니깐, 이벤트에 내가 신청 한번 해본 거지. 비싼 거야. ...",5
4,에피타이저 정말 좋아해. 그 것도 괜찮은 생각인 것 같애.,4


# Train / Valid / Test Split

In [6]:
from sklearn.model_selection import train_test_split

trval_X, test_X, trval_y, test_y = train_test_split(
    input_data[x_col].tolist(),
    input_data[y_col].tolist(),
    test_size=0.05,
    stratify=input_data[y_col],
    random_state=42
)

train_X, valid_X, train_y, valid_y = train_test_split(
    trval_X,
    trval_y,
    test_size=0.05,
    stratify=trval_y,
    random_state=42
)

print(f"train size: {len(train_X)}")
print(f"valid size: {len(valid_X)}")
print(f"test size : {len(test_X)}")

train size: 17484
valid size: 921
test size : 969


# Custom KoBERT Tokenizer

In [7]:
from konlpy.tag import Okt
from collections import Counter

# Custom KoBERT Tokenizer 정의
class CustomKoBERTTokenizer:
    def __init__(self, vocab_size=1000, min_freq=2):
        self.vocab_size = vocab_size
        self.min_freq = min_freq
        self.vocab = {}
        self.inv_vocab = {}
        self.unk_token = "[UNK]"
        self.pad_token = "[PAD]"
        self.cls_token = "[CLS]"
        self.sep_token = "[SEP]"
        self.special_tokens = [self.pad_token, self.cls_token, self.sep_token, self.unk_token]
        self.tokenizer = Okt()

    # Custom Tokenizer 학습 함수
    def train(self, corpus):
        tokens_all = []
        for sentence in corpus:
            morphs = [word for word, tag in self.tokenizer.pos(sentence, stem=True)]
            tokens_all.extend(morphs)

        counter = Counter(tokens_all)

        for i, token in enumerate(self.special_tokens):
            self.vocab[token] = i

        sorted_tokens = [token for token, freq in counter.most_common() if freq >= self.min_freq]

        for token in sorted_tokens[:self.vocab_size - len(self.special_tokens)]:
            if token not in self.vocab:
                self.vocab[token] = len(self.vocab)

        self.inv_vocab = {v: k for k, v in self.vocab.items()}

    #Tokenize / Padding 함수
    def encode(self, text):
        morphs = [word for word, tag in self.tokenizer.pos(text, stem=True)]

        token_ids = []
        for m in morphs:
            if m in self.vocab:
                token_ids.append(self.vocab[m])
            else:
                token_ids.append(self.vocab[self.unk_token])

        token_type_ids = [0] * len(token_ids)
        attention_mask = [1] * len(token_ids)

        return {
            "input_ids": token_ids,
            "token_type_ids": token_type_ids,
            "attention_mask": attention_mask
        }, morphs

    def pad_sequence(self, token_ids, max_length):
        pad_id = self.vocab[self.pad_token]
        if len(token_ids) > max_length:
            return token_ids[:max_length]
        else:
            return token_ids + [pad_id] * (max_length - len(token_ids))
    # 토크나이저 호출 함수
    def __call__(self, text, padding=None, truncation=None, return_tensors=None, max_length=None):
        encoded, tokens = self.encode(text)
        token_ids = encoded["input_ids"]

        if max_length is not None:
            token_ids = self.pad_sequence(token_ids, max_length)

        attention_mask = [1 if tid != self.vocab[self.pad_token] else 0 for tid in token_ids]
        token_type_ids = [0] * len(token_ids)

        encoded = {
            "input_ids": token_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids
        }

        if return_tensors == "pt":
            import torch
            encoded = {k: torch.tensor([v]) for k, v in encoded.items()}

        return encoded

    def decode(self, tokens):
        # tokens가 dict이면 input_ids 꺼내서 단어로 변환
        if isinstance(tokens, dict):
            input_ids = tokens.get('input_ids', [])
            words = [self.inv_vocab.get(i, self.unk_token) for i in input_ids]
            return " ".join(words)
        # tokens가 리스트면 단순 합치기
        elif isinstance(tokens, list):
            return " ".join(tokens)
        else:
            return str(tokens)


## Custom Tokenizer 학습(vocab 생성)

토크나이저 첫 생성 시 vocab 만들기 위해

문장 데이터 입력해서 학습시킴

In [8]:
corpus = input_data['발화문'].tolist()

custom_tokenizer = CustomKoBERTTokenizer(vocab_size=5000)

custom_tokenizer.train(corpus)

print("target vocab_size:", custom_tokenizer.vocab_size)
print("actual vocab size:", len(custom_tokenizer.vocab))

target vocab_size: 5000
actual vocab size: 2174


## 구현: Tokenizer 테스트

In [9]:
test_sentences = [
    "그는 밥을 먹는다",
    "빨리 먹는다",
    "맛있게 먹는다"
]

print("=== Testing word '먹는다' ===")
for sentence in test_sentences:
  #최소한 토큰 아이디는 반환해줘야함.
  encoded, tokens = custom_tokenizer.encode(sentence)

  print(f"\nText     : {sentence}")
  print(f"Tokens   : {tokens}")
  print(f"Token ids: {encoded['input_ids']}")
  print(f"Decoded  : {custom_tokenizer.decode(encoded)}")


=== Testing word '먹는다' ===

Text     : 그는 밥을 먹는다
Tokens   : ['그', '는', '밥', '을', '먹다']
Token ids: [57, 33, 747, 14, 101]
Decoded  : 그 는 밥 을 먹다

Text     : 빨리 먹는다
Tokens   : ['빨리', '먹다']
Token ids: [210, 101]
Decoded  : 빨리 먹다

Text     : 맛있게 먹는다
Tokens   : ['맛있다', '먹다']
Token ids: [343, 101]
Decoded  : 맛있다 먹다


## 구현: OOV 테스트

In [10]:
# Test with OOV words
oov_sentences = [
    "나는 아이스아메리카노를 마신다",  # "아이스아메리카노" might be OOV
    "그는 스마트폰으로 인터넷서핑을 한다"  # "스마트폰", "인터넷서핑" might be OOV
]

print("\n=== Testing OOV handling ===")

for sentence in oov_sentences:
    encoded, tokens = custom_tokenizer.encode(sentence)

    print(f"\nText     : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"Token ids: {encoded['input_ids']}")
    print(f"Decoded  : {custom_tokenizer.decode(encoded)}")
    # 보캡에 없는 것으로 나온다면 스페셜 토큰을 사용하여 unk 으로 처리해주면 됨.


=== Testing OOV handling ===

Text     : 나는 아이스아메리카노를 마신다
Tokens   : ['나', '는', '아이스', '아메리카노', '를', '말다']
Token ids: [7, 33, 3, 3, 17, 876]
Decoded  : 나 는 [UNK] [UNK] 를 말다

Text     : 그는 스마트폰으로 인터넷서핑을 한다
Tokens   : ['그', '는', '스마트폰', '으로', '인터넷', '서핑', '을', '하다']
Token ids: [57, 33, 3, 131, 1527, 3, 14, 5]
Decoded  : 그 는 [UNK] 으로 인터넷 [UNK] 을 하다


## tokenizer 로드

In [11]:
model_path = "drive/MyDrive/2025/KW/Model/"
model_id = "monologg/kobert"

In [12]:
from transformers import AutoTokenizer
kobert_tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=model_path, trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [13]:
kobert_tokenizer.vocab_size

8002

## Special Token 확인

In [14]:
print("All special tokens:    ", kobert_tokenizer.all_special_tokens)
print("All special token IDs: ", kobert_tokenizer.all_special_ids)
print("Special tokens map:    ", kobert_tokenizer.special_tokens_map)

All special tokens:     ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
All special token IDs:  [0, 3, 1, 2, 4]
Special tokens map:     {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


## kobert 토크나이저 & custom 토크나이저 비교

In [15]:
text = "그는 밥을 먹는다"

inputs = kobert_tokenizer(
    text,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
    max_length=20
)

inputs

{'input_ids': tensor([[   2, 1191, 2266, 7088, 2010, 5760, 5782,    3,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

In [16]:
inputs = custom_tokenizer(
    text,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
    max_length=20
)

inputs

{'input_ids': tensor([[ 57,  33, 747,  14, 101,   0,   0,   0,   0,   0,   0,   0,   0,   0,
            0,   0,   0,   0,   0,   0]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

In [17]:
txt = train_X[0]

print("TEXT:", txt)
print()

print("KoBERT")
print(kobert_tokenizer.encode(txt)[:10])
print()

print("Custom tokenizer")
encoded, tokens = custom_tokenizer.encode(txt)
print(tokens[:10])
print(encoded["input_ids"][:10])

TEXT: 그렇지. 경찰분들 진짜 고생 많으신 것 같애.

KoBERT
[2, 1205, 54, 975, 6416, 5931, 4368, 993, 6542, 517]

Custom tokenizer
['그렇다', '.', '경찰', '분들', '진짜', '고생', '많다', '것', '같다', '애']
[15, 4, 177, 1377, 96, 297, 94, 21, 19, 164]


## Dataset 정의

In [18]:
from torch.utils.data import Dataset

class KoBERTDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]
        label = self.labels[idx]

        inputs = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "label": torch.tensor(label)
        }

# DataLoader 생성

In [19]:
from torch.utils.data import DataLoader

batch_size = 64

train_dataset = KoBERTDataset(train_X, train_y, custom_tokenizer)
valid_dataset = KoBERTDataset(valid_X, valid_y, custom_tokenizer)
test_dataset = KoBERTDataset(test_X, test_y, custom_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

## KoBERT 모델 로드

In [20]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
from transformers import BertModel
bert = BertModel.from_pretrained(model_id, cache_dir=model_path, trust_remote_code=True)

Loading weights:   0%|          | 0/199 [00:10<?, ?it/s]

In [22]:
import torch.nn as nn

class KoBERTClassifier(nn.Module):
    def __init__(self, bert, num_classes, hidden_size=768, dropout=0.2):
        super(KoBERTClassifier, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        dropped = self.dropout(pooled)
        return self.classifier(dropped)

In [23]:
num_classes = len(df['y'].unique())
num_classes  # 7

7

In [24]:
model = KoBERTClassifier(bert, num_classes=num_classes).to(device)

## Train

In [25]:
from torch.optim import AdamW
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

# Optimizer and loss
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

epochs = 10

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss = train_loss / len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation
    model.eval()
    valid_loss = 0
    valid_preds = []
    valid_labels = []

    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            valid_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            valid_preds.extend(preds.cpu().numpy())
            valid_labels.extend(labels.cpu().numpy())

    valid_loss = valid_loss / len(valid_loader)
    valid_acc = accuracy_score(valid_labels, valid_preds)

    print(f"Epoch {epoch+1} - loss: {train_loss:.4f}  acc: {train_acc:.4f} | "
          f"val loss: {valid_loss:.4f}  val acc: {valid_acc:.4f}")

Epoch 1 - Training:   0%|          | 0/274 [00:00<?, ?it/s]

Epoch 1 - loss: 1.6386  acc: 0.3726 | val loss: 1.2901  val acc: 0.5537


Epoch 2 - Training:   0%|          | 0/274 [00:00<?, ?it/s]

Epoch 2 - loss: 0.9340  acc: 0.6859 | val loss: 0.6797  val acc: 0.7796


Epoch 3 - Training:   0%|          | 0/274 [00:00<?, ?it/s]

...

Epoch 8 - loss: 0.2189  acc: 0.9275 | val loss: 0.4209  val acc: 0.8740


Epoch 9 - Training:   0%|          | 0/274 [00:00<?, ?it/s]

Epoch 9 - loss: 0.1864  acc: 0.9376 | val loss: 0.4252  val acc: 0.8806


Epoch 10 - Training:   0%|          | 0/274 [00:00<?, ?it/s]

Epoch 10 - loss: 0.1647  acc: 0.9444 | val loss: 0.4265  val acc: 0.8730


## Evaluation

In [26]:
    model.eval()

    valid_preds = []
    valid_labels = []

    with torch.no_grad():

        for batch in valid_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)

            preds = torch.argmax(outputs, dim=1)

            valid_preds.extend(preds.cpu().numpy())
            valid_labels.extend(labels.cpu().numpy())

    valid_acc = accuracy_score(valid_labels, valid_preds)

    print(
        f"Epoch {epoch+1} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {valid_acc:.4f}"
    )

Epoch 10 | Train Acc: 0.9444 | Val Acc: 0.8730


In [27]:
# 예측할 문장 리스트
sample_sentences = [
    "정말 기분이 좋아!",
    "이게 뭐야, 너무 열받아!",
    "어떻게 해야 할지 모르겠어",
    "진짜 깜짝 놀랐어"
]

# 예측
model.eval()
for sentence in sample_sentences:
    # 토크나이즈 및 텐서화
    inputs = custom_tokenizer(sentence, return_tensors="pt", max_length=64, padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])
        pred = torch.argmax(outputs, dim=1).item()

    # 레이블 디코딩
    label_map_reverse = {v: k for k, v in label_map.items()}
    print(f"문장: \"{sentence}\" → 예측된 감정: {label_map_reverse[pred]}")


문장: "정말 기분이 좋아!" → 예측된 감정: happiness
문장: "이게 뭐야, 너무 열받아!" → 예측된 감정: disgust
문장: "어떻게 해야 할지 모르겠어" → 예측된 감정: neutral
문장: "진짜 깜짝 놀랐어" → 예측된 감정: surprise
